In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

In [3]:
data_path = r"C:\Users\Admin\data\train_processed_advanced.csv"

df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (336715, 54)


,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH,attack_class
0,0.0,1.619565,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
1,0.0,0.369565,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
2,0.0,-0.159420,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-1.0,0.0,DoS
3,0.0,0.681159,15.800388,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
4,0.0,0.561594,0.813953,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal


In [5]:
print(df.columns)

Index(['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
       'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
       'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
       'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'service_freq', 'protocol_type_icmp',
       'protocol_type_tcp', 'protocol_type_udp', 'flag_OTH', 'flag_REJ',
       'flag_RSTO', 'flag_RSTOS0', 'flag_RSTR', 'flag_S0', 'flag_S1',
       'flag_S2', 'flag_S3', 'flag_SF', 'fl

In [7]:
target_column = df.columns[-1]   # last column as target

X = df.drop(target_column, axis=1)
y = df[target_column]

print("Target column:", target_column)

Target column: attack_class


In [9]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

print("Numerical Columns:", len(num_cols))
print("Categorical Columns:", len(cat_cols))

Numerical Columns: 53
Categorical Columns: 0


In [10]:
num_pipeline = Pipeline([

    ("imputer", SimpleImputer(strategy="median")),  # fill missing values

    ("scaler", StandardScaler())  # scaling station

])

In [12]:
# Backward fill
X[num_cols] = X[num_cols].bfill()

# Forward fill
X[num_cols] = X[num_cols].ffill()

In [13]:
cat_pipeline = Pipeline([

    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("encoder", OneHotEncoder(handle_unknown="ignore"))

])

In [14]:
preprocessor = ColumnTransformer([

    ("num", num_pipeline, num_cols),

    ("cat", cat_pipeline, cat_cols)

])

In [15]:
feature_selector = SelectFromModel(

    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

)

In [16]:
classifier = RandomForestClassifier(

    n_estimators=200,
    random_state=42,
    n_jobs=-1

)

In [17]:
model_pipeline = Pipeline([

    ("preprocessing", preprocessor),

    ("feature_selection", feature_selector),

    ("classifier", classifier)

])

In [18]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42

)

In [19]:
model_pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [22]:
y_pred = model_pipeline.predict(X_test)

In [23]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))

Accuracy: 0.9995248206940588

Classification Report:

              precision    recall  f1-score   support

         DoS       1.00      1.00      1.00     13340
       Probe       1.00      1.00      1.00     13499
         R2L       1.00      1.00      1.00     13629
         U2R       1.00      1.00      1.00     13368
      normal       1.00      1.00      1.00     13507

    accuracy                           1.00     67343
   macro avg       1.00      1.00      1.00     67343
weighted avg       1.00      1.00      1.00     67343

